In [10]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
================================================================================
АВТОМАТИЗИРОВАННЫЙ МОНИТОРИНГ И РЕАГИРОВАНИЕ НА УГРОЗЫ
================================================================================
Дисциплина: Программирование на Python
Темы: 7-14
Автор: [Ваше имя]
Дата: 2026

Описание:
    Скрипт для анализа логов безопасности (Suricata и Nessus) с обогащением 
    данных через VirusTotal API, выявлением угроз, имитацией реагирования 
    и формированием отчетов с визуализацией.

Источники данных:
    1. Логи Suricata (alerts-only.json) - события сетевой безопасности
    2. Логи Nessus (log_example.txt) - SSH атаки и уязвимости
    3. API VirusTotal - проверка подозрительных IP-адресов

Функциональность:
    - Загрузка и парсинг логов различных форматов
    - Анализ и классификация угроз по критичности
    - Проверка IP через VirusTotal API
    - Имитация блокировки подозрительных IP
    - Генерация отчетов в JSON и CSV
    - Визуализация результатов в PNG графиках
================================================================================
"""

# ========================================================================
# БЛОК 1: ИМПОРТЫ И НАСТРОЙКА ОКРУЖЕНИЯ
# ========================================================================

import os  # Для работы с файловой системой и переменными окружения
import json  # Для парсинга и создания JSON файлов
import re  # Для работы с регулярными выражениями (парсинг логов)
import requests  # Для отправки HTTP запросов к API VirusTotal
import pandas as pd  # Для обработки и анализа табличных данных
import matplotlib.pyplot as plt  # Для создания графиков и визуализации
from datetime import datetime  # Для работы с датами и временем
from collections import Counter  # Для подсчета частоты элементов
import warnings  # Для управления предупреждениями
warnings.filterwarnings('ignore')  # Отключаем предупреждения для чистоты вывода

# ========================================================================
# БЛОК 2: НАСТРОЙКА ПАРАМЕТРОВ
# ========================================================================

# Установка API ключа VirusTotal (ОБЯЗАТЕЛЬНО ЗАМЕНИТЕ НА СВОЙ!)
# Получение ключа из переменной окружения (безопасный способ)
VT_API_KEY = os.environ.get('VT_API_KEY')
if not VT_API_KEY:
    # Если ключ не найден в переменных окружения, устанавливаем напрямую
    # ВНИМАНИЕ: Для продакшена используйте переменные окружения!
    os.environ['VT_API_KEY'] = 'API_KEY'
    VT_API_KEY = os.environ['VT_API_KEY']

print("=" * 70)
print("🔐 АВТОМАТИЗИРОВАННЫЙ МОНИТОРИНГ И РЕАГИРОВАНИЕ НА УГРОЗЫ")
print("=" * 70)
print(f"✅ Библиотеки импортированы успешно")
print(f"🔑 API ключ загружен: {VT_API_KEY[:10]}... (первые 10 символов)")

# Определяем пути к файлам
SCRIPT_DIR = os.getcwd()  # Текущая рабочая директория
SURICATA_PATH = os.path.join(SCRIPT_DIR, "alerts-only.json")  # Путь к логам Suricata
NESSUS_PATH = os.path.join(SCRIPT_DIR, "log_example.txt")  # Путь к логам Nessus
REPORT_PATH = os.path.join(SCRIPT_DIR, "threat_report.json")  # Путь для JSON отчета
CSV_PATH = os.path.join(SCRIPT_DIR, "threat_analysis.csv")  # Путь для CSV отчета
GRAPH_PATH = os.path.join(SCRIPT_DIR, "threat_visualization.png")  # Путь для основного графика
CVSS_GRAPH_PATH = os.path.join(SCRIPT_DIR, "cvss_distribution.png")  # Путь для графика атак

# Параметры анализа (можно настраивать под свои нужды)
SUSPICIOUS_SSH_PATTERNS = [
    r'invalid user',  # Паттерн для невалидных пользователей
    r'failed password',  # Паттерн для неудачных попыток входа
    r'connection reset',  # Паттерн для сброса соединения
    r'log4shell',  # Паттерн для Log4Shell атаки
    r'\$\{jndi:'  # Паттерн для JNDI инъекций
]
LOG4SHELL_PATTERN = r'\$\{jndi:.*\}'  # Регулярка для Log4Shell
SSH_BRUTE_FORCE_THRESHOLD = 3  # Порог для определения брутфорс атак

# Проверяем наличие файлов с данными
print("\n📁 Проверка файлов:")
print(f"   Suricata: {'✅' if os.path.exists(SURICATA_PATH) else '❌'} {os.path.basename(SURICATA_PATH)}")
print(f"   Nessus: {'✅' if os.path.exists(NESSUS_PATH) else '❌'} {os.path.basename(NESSUS_PATH)}")
print()

# ========================================================================
# БЛОК 3: КЛАСС ДЛЯ АНАЛИЗА УГРОЗ
# ========================================================================

class ThreatAnalyzer:
    """
    Класс для анализа угроз безопасности из различных источников.
    
    Атрибуты:
        suricata_alerts (list): Список алертов из Suricata
        nessus_events (list): Список событий из Nessus
        threats (list): Найденные угрозы
        vt_cache (dict): Кэш результатов VirusTotal
        stats (dict): Статистика анализа
    """
    
    def __init__(self):
        """Инициализация анализатора угроз"""
        self.suricata_alerts = []  # Список алертов Suricata
        self.nessus_events = []    # Список событий Nessus
        self.threats = []           # Список найденных угроз
        self.vt_cache = {}          # Кэш для результатов VirusTotal
        self.stats = {              # Статистика анализа
            'suricata_alerts': 0,
            'nessus_events': 0,
            'vt_checks': 0,
            'malicious_ips': 0
        }
        
    def load_suricata_alerts(self):
        """
        Загрузка алертов Suricata из JSON файла.
        Поддерживает два формата:
        - JSON массив [ {...}, {...} ]
        - JSON Lines (каждая строка - отдельный JSON объект)
        """
        print("\n📥 Загрузка Suricata алертов...")
        
        # Проверяем существование файла
        if not os.path.exists(SURICATA_PATH):
            print("   ⚠️ Файл не найден, пропускаем...")
            return
            
        try:
            # Открываем файл для чтения
            with open(SURICATA_PATH, 'r', encoding='utf-8') as f:
                content = f.read().strip()
                
                # Проверяем формат файла
                if content.startswith('['):
                    # Формат JSON массива
                    self.suricata_alerts = json.loads(content)
                    print(f"   📄 Обнаружен формат JSON массива")
                else:
                    # Формат JSON Lines (построчный)
                    self.suricata_alerts = []
                    for line in content.split('\n'):
                        if line.strip():
                            try:
                                self.suricata_alerts.append(json.loads(line.strip()))
                            except json.JSONDecodeError:
                                continue
                    print(f"   📄 Обнаружен формат JSON Lines")
            
            # Сохраняем статистику
            self.stats['suricata_alerts'] = len(self.suricata_alerts)
            print(f"   ✅ Загружено {self.stats['suricata_alerts']} алертов")
            
            # Показываем пример алерта для отладки
            if self.suricata_alerts:
                print(f"   📋 Пример алерта: {json.dumps(self.suricata_alerts[0], indent=2)[:200]}...")
                
        except Exception as e:
            print(f"   ❌ Ошибка загрузки: {e}")
    
    def parse_nessus_logs(self):
        """
        Парсинг логов Nessus в формате syslog.
        Формат строки:
        "Jan 20 22:54:47 demohost001 sshd[1762]: Connection reset by 192.168.0.15 port 52678 [preauth]"
        """
        print("\n📥 Парсинг Nessus логов...")
        
        # Проверяем существование файла
        if not os.path.exists(NESSUS_PATH):
            print("   ⚠️ Файл не найден, пропускаем...")
            return
            
        # Регулярное выражение для парсинга syslog формата
        # Группы: 1 - timestamp, 2 - host, 3 - service, 4 - pid, 5 - message
        log_pattern = re.compile(
            r'(\w+\s+\d+\s+\d+:\d+:\d+)\s+(\S+)\s+(\S+)\[(\d+)\]:\s+(.*)'
        )
        
        try:
            # Открываем файл для чтения
            with open(NESSUS_PATH, 'r', encoding='utf-8') as f:
                for line in f:
                    line = line.strip()
                    if not line:
                        continue  # Пропускаем пустые строки
                        
                    # Применяем регулярное выражение
                    match = log_pattern.match(line)
                    if match:
                        # Извлекаем компоненты лога
                        timestamp, host, service, pid, message = match.groups()
                        
                        # Извлекаем IP-адрес из сообщения
                        ip_match = re.search(r'(\d+\.\d+\.\d+\.\d+)', message)
                        ip = ip_match.group(1) if ip_match else None
                        
                        # Определяем тип события на основе ключевых слов
                        event_type = 'normal'  # По умолчанию нормальное событие
                        
                        if re.search(r'invalid user', message, re.IGNORECASE):
                            event_type = 'invalid_user'
                        elif re.search(r'failed password', message, re.IGNORECASE):
                            event_type = 'failed_password'
                        elif re.search(r'connection reset', message, re.IGNORECASE):
                            event_type = 'connection_reset'
                        elif re.search(r'log4shell|\$\{jndi:', message, re.IGNORECASE):
                            event_type = 'log4shell_attempt'
                        
                        # Определяем уровень критичности
                        severity = 'LOW'
                        if event_type == 'log4shell_attempt':
                            severity = 'CRITICAL'  # Log4Shell - критическая уязвимость
                        elif event_type == 'failed_password':
                            severity = 'HIGH'      # Неудачные попытки входа - высокий риск
                        elif event_type == 'invalid_user':
                            severity = 'MEDIUM'    # Невалидные пользователи - средний риск
                        
                        # Сохраняем событие
                        self.nessus_events.append({
                            'timestamp': timestamp,
                            'host': host,
                            'service': service,
                            'pid': pid,
                            'message': message,
                            'ip': ip,
                            'event_type': event_type,
                            'severity': severity
                        })
            
            # Сохраняем статистику
            self.stats['nessus_events'] = len(self.nessus_events)
            print(f"   ✅ Загружено {self.stats['nessus_events']} событий")
            
            # Выводим статистику по типам событий
            event_types = Counter([e['event_type'] for e in self.nessus_events])
            print("   📊 Распределение по типам:")
            for e_type, count in event_types.most_common():
                print(f"      - {e_type}: {count}")
                
        except Exception as e:
            print(f"   ❌ Ошибка парсинга: {e}")
    
    def analyze_threats(self):
        """
        Анализ загруженных данных и выявление угроз.
        Комбинирует данные из Suricata и Nessus, классифицирует угрозы.
        """
        print("\n🔍 Анализ угроз...")
        
        # Анализ Suricata алертов (ограничиваем 200 для производительности)
        for alert in self.suricata_alerts[:200]:
            # Базовая информация об угрозе
            threat = {
                'source': 'Suricata IDS',
                'timestamp': alert.get('timestamp', datetime.now().isoformat()),
                'severity': 'HIGH',  # По умолчанию HIGH для IDS алертов
                'ip': alert.get('src_ip', alert.get('source_ip', 'unknown')),
                'dest_ip': alert.get('dest_ip', alert.get('destination_ip', 'unknown')),
                'threat_type': 'IDS Alert'
            }
            
            # Извлекаем сигнатуру алерта
            if 'alert' in alert and isinstance(alert['alert'], dict):
                threat['details'] = alert['alert'].get('signature', 'Unknown signature')
                threat['category'] = alert['alert'].get('category', 'unknown')
            else:
                threat['details'] = 'Suricata alert'
            
            self.threats.append(threat)
        
        # Анализ Nessus событий
        ip_attempts = Counter()  # Счетчик попыток с каждого IP
        
        for event in self.nessus_events:
            # Учитываем только подозрительные события
            if event['event_type'] != 'normal' and event['ip']:
                ip_attempts[event['ip']] += 1
                
                # Добавляем угрозу
                self.threats.append({
                    'source': 'SSH Logs',
                    'timestamp': event['timestamp'],
                    'severity': event['severity'],
                    'ip': event['ip'],
                    'threat_type': f'SSH_{event["event_type"]}',
                    'details': event['message'][:150] + '...' if len(event['message']) > 150 else event['message']
                })
        
        # Выявляем брутфорс атаки (множественные попытки с одного IP)
        for ip, attempts in ip_attempts.items():
            if attempts > SSH_BRUTE_FORCE_THRESHOLD:
                self.threats.append({
                    'source': 'SSH Logs',
                    'timestamp': 'multiple',
                    'severity': 'HIGH',
                    'ip': ip,
                    'threat_type': 'SSH_Brute_Force',
                    'details': f'Multiple attack attempts ({attempts}) from same IP'
                })
        
        print(f"   ✅ Найдено {len(self.threats)} потенциальных угроз")
    
    def check_ip_virustotal(self, ip):
        """
        Проверка IP-адреса через VirusTotal API.
        
        Args:
            ip (str): IP-адрес для проверки
            
        Returns:
            dict: Результаты проверки или None при ошибке
        """
        # Проверяем кэш
        if ip in self.vt_cache:
            return self.vt_cache.get(ip)
        
        # Пропускаем невалидные IP
        if not ip or ip == 'unknown':
            return None
        
        # Пропускаем локальные IP (не проверяем через VT)
        if ip.startswith(('192.168.', '10.', '127.', '172.16.')):
            return None
            
        # Формируем запрос к API
        url = f"https://www.virustotal.com/api/v3/ip_addresses/{ip}"
        headers = {"x-apikey": VT_API_KEY}
        
        try:
            # Отправляем запрос
            self.stats['vt_checks'] += 1
            response = requests.get(url, headers=headers, timeout=10)
            
            if response.status_code == 200:
                # Парсим ответ
                data = response.json()
                stats = data.get('data', {}).get('attributes', {}).get('last_analysis_stats', {})
                result = {
                    'malicious': stats.get('malicious', 0),
                    'suspicious': stats.get('suspicious', 0),
                    'harmless': stats.get('harmless', 0),
                    'undetected': stats.get('undetected', 0)
                }
                # Сохраняем в кэш
                self.vt_cache[ip] = result
                
                # Обновляем статистику
                if result['malicious'] > 0 or result['suspicious'] > 0:
                    self.stats['malicious_ips'] += 1
                    
                return result
            else:
                # Ошибка API
                return None
                
        except Exception as e:
            # Ошибка соединения
            return None
    
    def enrich_with_virustotal(self):
        """
        Обогащение данных через VirusTotal API.
        Проверяет все уникальные IP из найденных угроз.
        """
        print("\n🔍 Обогащение данных через VirusTotal API...")
        
        # Собираем уникальные IP из угроз
        ips_to_check = set()
        for threat in self.threats:
            if 'ip' in threat and threat['ip'] and threat['ip'] != 'unknown':
                ips_to_check.add(threat['ip'])
        
        print(f"   Найдено {len(ips_to_check)} уникальных IP для проверки")
        
        # Проверяем каждый IP (ограничиваем 10 для экономии времени)
        for i, ip in enumerate(list(ips_to_check)[:10], 1):
            print(f"   [{i}/{min(10, len(ips_to_check))}] Проверка {ip}...", end=' ')
            vt_result = self.check_ip_virustotal(ip)
            
            # Если IP вредоносный, добавляем отдельную угрозу
            if vt_result and (vt_result['malicious'] > 0 or vt_result['suspicious'] > 0):
                print(f"🔴 Найден в VT!")
                self.threats.append({
                    'source': 'VirusTotal',
                    'timestamp': datetime.now().isoformat(),
                    'severity': 'CRITICAL' if vt_result['malicious'] > 3 else 'HIGH',
                    'ip': ip,
                    'threat_type': 'Malicious_IP',
                    'details': f'VT Stats: {vt_result["malicious"]} malicious, {vt_result["suspicious"]} suspicious',
                    'vt_stats': vt_result
                })
            else:
                print("✅ Чист")
    
    def run(self):
        """
        Запуск полного цикла анализа.
        
        Returns:
            list: Список найденных угроз
        """
        print("\n" + "=" * 70)
        print("🚀 ЗАПУСК АНАЛИЗА БЕЗОПАСНОСТИ")
        print("=" * 70)
        
        # Этап 1: Загрузка данных
        self.load_suricata_alerts()
        self.parse_nessus_logs()
        
        # Этап 2: Анализ угроз
        self.analyze_threats()
        
        # Этап 3: Обогащение через VirusTotal
        self.enrich_with_virustotal()
        
        # Итоги анализа
        print("\n" + "=" * 70)
        print("📊 ИТОГИ АНАЛИЗА:")
        print(f"   - Всего угроз: {len(self.threats)}")
        print(f"   - Проверено IP в VT: {self.stats['vt_checks']}")
        print(f"   - Найдено вредоносных IP: {self.stats['malicious_ips']}")
        print("=" * 70)
        
        return self.threats


# ========================================================================
# БЛОК 4: КЛАСС ДЛЯ РЕАГИРОВАНИЯ НА УГРОЗЫ
# ========================================================================

class IncidentResponder:
    """
    Класс для имитации реагирования на найденные угрозы.
    
    Атрибуты:
        threats (list): Список угроз
        blocked_ips (set): Множество заблокированных IP
        notifications (list): Список отправленных уведомлений
    """
    
    def __init__(self, threats):
        """
        Инициализация обработчика инцидентов.
        
        Args:
            threats (list): Список найденных угроз
        """
        self.threats = threats
        self.blocked_ips = set()
        self.notifications = []
        
    def respond(self):
        """
        Реагирование на угрозы (имитация).
        Блокирует IP с критическими и высокими угрозами.
        
        Returns:
            dict: Информация о реагировании
        """
        print("\n" + "=" * 70)
        print("🚨 РЕАГИРОВАНИЕ НА УГРОЗЫ")
        print("=" * 70)
        
        # Отбираем критические и высокие угрозы
        critical_threats = [t for t in self.threats if t.get('severity') in ['CRITICAL', 'HIGH']]
        
        if not critical_threats:
            print("✅ Критических угроз не обнаружено")
            return {'blocked_ips': [], 'notifications': []}
        
        print(f"⚠️ Обнаружено {len(critical_threats)} критических угроз")
        
        # Обрабатываем каждую угрозу
        for threat in critical_threats:
            ip = threat.get('ip')
            
            # Блокируем IP, если он валидный и еще не заблокирован
            if ip and ip != 'unknown' and ip not in self.blocked_ips:
                print(f"\n🔒 БЛОКИРОВКА IP: {ip}")
                print(f"   Тип угрозы: {threat['threat_type']}")
                print(f"   Критичность: {threat['severity']}")
                print(f"   Детали: {threat['details'][:100]}")
                
                self.blocked_ips.add(ip)
                
                # Создаем уведомление (имитация)
                notification = {
                    'timestamp': datetime.now().isoformat(),
                    'ip': ip,
                    'threat_type': threat['threat_type'],
                    'severity': threat['severity']
                }
                self.notifications.append(notification)
                print(f"   📧 Уведомление отправлено администратору")
        
        # Итоги реагирования
        print("\n" + "=" * 70)
        print(f"📊 ИТОГИ РЕАГИРОВАНИЯ:")
        print(f"   - Заблокировано IP: {len(self.blocked_ips)}")
        print(f"   - Отправлено уведомлений: {len(self.notifications)}")
        
        return {
            'blocked_ips': list(self.blocked_ips),
            'notifications': self.notifications
        }


# ========================================================================
# БЛОК 5: КЛАСС ДЛЯ ФОРМИРОВАНИЯ ОТЧЕТОВ
# ========================================================================

class ReportGenerator:
    """
    Класс для генерации отчетов и визуализации результатов.
    
    Атрибуты:
        threats (list): Список угроз
        response_data (dict): Данные о реагировании
    """
    
    def __init__(self, threats, response_data):
        """
        Инициализация генератора отчетов.
        
        Args:
            threats (list): Список найденных угроз
            response_data (dict): Результаты реагирования
        """
        self.threats = threats
        self.response_data = response_data
        
    def save_reports(self):
        """
        Сохранение отчетов в JSON и CSV форматах.
        
        Returns:
            dict: Сгенерированный отчет
        """
        print("\n📄 Сохранение отчетов...")
        
        # Подсчет статистики
        severity_counts = Counter([t.get('severity', 'UNKNOWN') for t in self.threats])
        source_counts = Counter([t.get('source', 'UNKNOWN') for t in self.threats])
        type_counts = Counter([t.get('threat_type', 'UNKNOWN') for t in self.threats])
        
        # Формируем JSON отчет
        report = {
            'generated': datetime.now().isoformat(),
            'total_threats': len(self.threats),
            'blocked_ips': self.response_data['blocked_ips'],
            'statistics': {
                'by_severity': dict(severity_counts),
                'by_source': dict(source_counts),
                'by_type': dict(type_counts.most_common(10))
            },
            'threats': self.threats  # Полный список угроз
        }
        
        # Сохраняем JSON
        with open(REPORT_PATH, 'w', encoding='utf-8') as f:
            json.dump(report, f, indent=2, ensure_ascii=False)
        
        # Сохраняем CSV (упрощенная версия без вложенных структур)
        df = pd.DataFrame(self.threats)
        if 'vt_stats' in df.columns:
            df = df.drop('vt_stats', axis=1)  # Удаляем сложное поле для CSV
        df.to_csv(CSV_PATH, index=False, encoding='utf-8')
        
        print(f"   ✅ JSON отчет: {os.path.basename(REPORT_PATH)}")
        print(f"   ✅ CSV отчет: {os.path.basename(CSV_PATH)}")
        
        return report
    
    def create_visualizations(self):
        """
        Создание графиков для визуализации результатов.
        Сохраняет два PNG файла:
        - threat_visualization.png - комплексный график
        - cvss_distribution.png - график активности SSH атак
        """
        print("\n📊 Создание визуализаций...")
        
        if not self.threats:
            print("   ⚠️ Нет данных для визуализации")
            return
        
        df = pd.DataFrame(self.threats)
        
        # ========== ГРАФИК 1: Комплексный (4 в 1) ==========
        fig, axes = plt.subplots(2, 2, figsize=(16, 12))
        fig.suptitle('Анализ угроз информационной безопасности', fontsize=16, fontweight='bold')
        
        # 1. Распределение по критичности
        if 'severity' in df.columns:
            severity_order = ['CRITICAL', 'HIGH', 'MEDIUM', 'LOW', 'UNKNOWN']
            sev_counts = df['severity'].value_counts()
            colors = {
                'CRITICAL': '#8B0000',  # Темно-красный
                'HIGH': '#FF4444',        # Красный
                'MEDIUM': '#FFA500',      # Оранжевый
                'LOW': '#FFD700',          # Желтый
                'UNKNOWN': '#808080'       # Серый
            }
            bar_colors = [colors.get(s, '#808080') for s in sev_counts.index]
            
            axes[0,0].bar(sev_counts.index, sev_counts.values, color=bar_colors, edgecolor='black')
            axes[0,0].set_title('Угрозы по уровню критичности', fontsize=12, fontweight='bold')
            axes[0,0].set_ylabel('Количество')
            axes[0,0].grid(True, alpha=0.3)
            
            # Добавляем значения на столбцы
            for i, (idx, val) in enumerate(sev_counts.items()):
                axes[0,0].text(i, val, str(val), ha='center', va='bottom', fontweight='bold')
        
        # 2. Топ источников угроз
        if 'source' in df.columns:
            src_counts = df['source'].value_counts().head(5)
            axes[0,1].barh(src_counts.index, src_counts.values, color='#4CAF50', edgecolor='black')
            axes[0,1].set_title('Топ-5 источников угроз', fontsize=12, fontweight='bold')
            axes[0,1].set_xlabel('Количество')
            axes[0,1].grid(True, alpha=0.3)
            
            # Добавляем значения
            for i, (idx, val) in enumerate(src_counts.items()):
                axes[0,1].text(val, i, f' {val}', va='center')
        
        # 3. Топ подозрительных IP
        if 'ip' in df.columns:
            ip_counts = df[df['ip'] != 'unknown']['ip'].value_counts().head(5)
            if not ip_counts.empty:
                axes[1,0].bar(range(len(ip_counts)), ip_counts.values, color='#FF6B6B', edgecolor='black')
                axes[1,0].set_xticks(range(len(ip_counts)))
                axes[1,0].set_xticklabels(ip_counts.index, rotation=45, ha='right')
                axes[1,0].set_title('Топ-5 подозрительных IP-адресов', fontsize=12, fontweight='bold')
                axes[1,0].set_ylabel('Количество')
                axes[1,0].grid(True, alpha=0.3)
        
        # 4. Типы угроз (круговая диаграмма)
        if 'threat_type' in df.columns:
            type_counts = df['threat_type'].value_counts().head(5)
            colors = plt.cm.Set3(range(len(type_counts)))
            axes[1,1].pie(type_counts.values, labels=type_counts.index, autopct='%1.1f%%',
                         colors=colors, startangle=90)
            axes[1,1].set_title('Топ-5 типов угроз', fontsize=12, fontweight='bold')
        
        plt.tight_layout()
        plt.savefig(GRAPH_PATH, dpi=100, bbox_inches='tight')
        plt.close()
        
        print(f"   ✅ Основной график: {os.path.basename(GRAPH_PATH)}")
        
        # ========== ГРАФИК 2: Активность SSH атак ==========
        ssh_events = [t for t in self.threats if t.get('source') == 'SSH Logs']
        if ssh_events:
            fig2, ax = plt.subplots(figsize=(12, 6))
            
            # Извлекаем часы из временных меток
            times = []
            for event in ssh_events:
                if event['timestamp'] != 'multiple':
                    try:
                        # Пробуем извлечь час из timestamp
                        time_str = event['timestamp']
                        # Формат может быть разным, пробуем разные варианты
                        if ' ' in time_str:
                            hour = time_str.split()[1][:2]  # "22:54:47" -> "22"
                        else:
                            hour = '00'
                        times.append(hour)
                    except (IndexError, AttributeError):
                        continue
            
            if times:
                hour_counts = Counter(times)
                hours = sorted(hour_counts.keys())
                counts = [hour_counts[h] for h in hours]
                
                # Строим график
                ax.plot(hours, counts, marker='o', linewidth=2, markersize=8, color='#FF4444')
                ax.fill_between(hours, counts, alpha=0.3, color='#FF4444')
                ax.set_title('Активность SSH атак по часам', fontsize=14, fontweight='bold')
                ax.set_xlabel('Час')
                ax.set_ylabel('Количество атак')
                ax.grid(True, alpha=0.3)
                
                # Добавляем значения на график
                for i, (h, c) in enumerate(zip(hours, counts)):
                    ax.text(h, c + 5, str(c), ha='center', fontweight='bold')
                
                plt.tight_layout()
                plt.savefig(CVSS_GRAPH_PATH, dpi=100, bbox_inches='tight')
                plt.close()
                print(f"   ✅ График атак: {os.path.basename(CVSS_GRAPH_PATH)}")


# ========================================================================
# БЛОК 6: ОСНОВНАЯ ПРОГРАММА
# ========================================================================

def main():
    """
    Главная функция программы.
    Выполняет все этапы анализа и формирует отчеты.
    """
    print("\n" + "=" * 70)
    print("🔐 АВТОМАТИЗИРОВАННЫЙ МОНИТОРИНГ И РЕАГИРОВАНИЕ НА УГРОЗЫ")
    print("=" * 70)
    
    # Этап 1-2: Анализ данных
    print("\n📊 ЭТАП 1-2: СБОР И АНАЛИЗ ДАННЫХ")
    analyzer = ThreatAnalyzer()
    threats = analyzer.run()
    
    # Этап 3: Реагирование
    print("\n🚨 ЭТАП 3: РЕАГИРОВАНИЕ НА УГРОЗЫ")
    responder = IncidentResponder(threats)
    response = responder.respond()
    
    # Этап 4: Отчеты и визуализация
    print("\n📈 ЭТАП 4: ФОРМИРОВАНИЕ ОТЧЕТОВ")
    reporter = ReportGenerator(threats, response)
    reporter.save_reports()
    reporter.create_visualizations()
    
    # Финальный вывод
    print("\n" + "=" * 70)
    print("✅ РАБОТА ЗАВЕРШЕНА УСПЕШНО")
    print("=" * 70)
    print(f"\n📁 Созданные файлы в папке: {SCRIPT_DIR}")
    print(f"   - {os.path.basename(REPORT_PATH)} (JSON отчет)")
    print(f"   - {os.path.basename(CSV_PATH)} (CSV отчет)")
    print(f"   - {os.path.basename(GRAPH_PATH)} (Визуализация)")
    if os.path.exists(CVSS_GRAPH_PATH):
        print(f"   - {os.path.basename(CVSS_GRAPH_PATH)} (График атак)")
    
    # Дополнительная статистика
    print("\n📊 ФИНАЛЬНАЯ СТАТИСТИКА:")
    df = pd.DataFrame(threats)
    print(f"   Всего угроз: {len(df)}")
    if 'severity' in df.columns:
        print("\n   Распределение по критичности:")
        for severity, count in df['severity'].value_counts().items():
            print(f"      {severity}: {count}")
    
    print("\n" + "=" * 70)


# ========================================================================
# БЛОК 7: ЗАПУСК ПРОГРАММЫ
# ========================================================================

if __name__ == "__main__":
    """
    Точка входа в программу.
    Запускается только при прямом вызове скрипта.
    """
    main()


# ========================================================================
# КОНЕЦ СКРИПТА
# ========================================================================

🔐 АВТОМАТИЗИРОВАННЫЙ МОНИТОРИНГ И РЕАГИРОВАНИЕ НА УГРОЗЫ
✅ Библиотеки импортированы успешно
🔑 API ключ загружен: c7cce2f13b... (первые 10 символов)

📁 Проверка файлов:
   Suricata: ✅ alerts-only.json
   Nessus: ✅ log_example.txt


🔐 АВТОМАТИЗИРОВАННЫЙ МОНИТОРИНГ И РЕАГИРОВАНИЕ НА УГРОЗЫ

📊 ЭТАП 1-2: СБОР И АНАЛИЗ ДАННЫХ

🚀 ЗАПУСК АНАЛИЗА БЕЗОПАСНОСТИ

📥 Загрузка Suricata алертов...
   📄 Обнаружен формат JSON массива
   ✅ Загружено 136 алертов
   📋 Пример алерта: {
  "timestamp": "2015-03-29T11:01:38.221126-0600",
  "flow_id": 1959620435402694,
  "pcap_cnt": 98966,
  "event_type": "alert",
  "src_ip": "222.186.56.46",
  "src_port": 4904,
  "dest_ip": "192.168....

📥 Парсинг Nessus логов...
   ✅ Загружено 1012 событий
   📊 Распределение по типам:
      - invalid_user: 572
      - normal: 264
      - connection_reset: 105
      - failed_password: 67
      - log4shell_attempt: 4

🔍 Анализ угроз...
   ✅ Найдено 885 потенциальных угроз

🔍 Обогащение данных через VirusTotal API...
   Найдено 